# MGMT47400 Predictive Analytics: End-to-End Workflow Churn Exit Prediction

**Professor Davi Moreira**




This notebook presents an **end-to-end predictive modeling workflow**, using our course competition as a practical example.

Because the competition requires reproducibility, we set a fixed random seed (**SEED = 474**). The competition also provides a `test.csv` file without outcome labels. For this reason, the `test.csv` will only be used at the very end of the process to generate our final prediction output (`.csv` file) after training our selected model over the entire training set.

The modeling process follows this structure:

* The original `train.csv` is used for both **model training** and **model evaluation** (via cross-validation).
* Once the best-performing model is selected, it is retrained on the **entire training set** before generating predictions for `test.csv`.

In other contexts, where only a single dataset is available (and no separate test file exists), we typically consider two strategies:

1. **Train/Test Split with Cross-Validation**: Divide the dataset into training and validation subsets. Use cross-validation to train and evaluate candidate models, then select the one with the lowest estimated test error.

2. **Three-Way Split (Train/Validation/Test)**: Partition the dataset into three subsets. Use training and validation data with cross-validation to tune and compare models, and keep the test subset entirely unseen for the final performance estimate.

Our workflow will be organized in the following sequence. For each step, we will also provide an AI-prompt suggestion that can be used to request the corresponding Python code from an AI tool.

- Step 0 — Setup, configuration, and imports
- Step 1 — Data loading and sanity checks
- Step 2 — Exploratory Data Analysis (EDA)
- Step 3 — Data Preparation and Feature Engineering
- Step 4 — Basic Model
- Step 5 — More Complex Models
- Step 6 — Compare models with visualization
- Step 7 — Model selection and rationale
- Step 8 — Final training on the full training data
- Step 9 — Predict on the new observations (test set) and export the submission `.csv` file
- Step 99 — Reporting and reproducibility

## Step 0 — Setup, configuration, and imports

## **AI-Prompt**

Generate well-structured Python code that accomplishes the following:

1. **Import Libraries**: Import `pandas`, `numpy`, `matplotlib.pyplot`, and key modules from `sklearn` (`model_selection`, `metrics`, `preprocessing`, `compose`, `pipeline`, `linear_model`, `feature_selection`), as well as `plotly.express`.

2. **Reproducibility**: Define `SEED = 474` and set all relevant global random seeds (NumPy, Python’s `random`, and scikit-learn if applicable).

3. **Display Settings**: Configure pandas display options for improved readability.

4. **Version Check**: Print out the version numbers of all imported libraries.

5. **Documentation**: Include clear comments and explanations for every step, written for someone new to Python and data science.

Ensure the code is clean, beginner-friendly, and ready to be executed in a Jupyter Notebook.


In [8]:
# This cell sets up the environment for our data analysis and modeling.

# 1. Import Libraries
# We import the necessary libraries for data manipulation, visualization, and machine learning.
# pandas is for data handling and analysis.
import pandas as pd
# numpy is for numerical operations.
import numpy as np
# matplotlib.pyplot is for creating static plots.
import matplotlib.pyplot as plt
# plotly.express is for creating interactive plots easily.
import plotly.express as px
# We import specific modules from scikit-learn for machine learning tasks.
from sklearn import __version__ as sklearn_version
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, roc_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SequentialFeatureSelector
import joblib # For saving and loading models
import sys # Import sys to check Python version


# Check if plotly is available, otherwise set to None for version check
try:
    import plotly
    _PLOTLY_AVAILABLE = True
except ImportError:
    _PLOTLY_AVAILABLE = False
    plotly = None


# 2. Reproducibility
# Setting a fixed random seed ensures that our results are reproducible.
# This is important for competitions and for comparing different model runs.
SEED = 474
np.random.seed(SEED)
import random
random.seed(SEED)
# Some scikit-learn functions also benefit from a global random state.
# This is often handled within specific function calls, but setting it globally where possible is good practice.
# For pipelines and models, we will pass the random_state parameter explicitly.

# 3. Display Settings
# Configure pandas options to display more rows and columns.
# This helps when inspecting dataframes.
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000) # Adjust display width

# 4. Version Check
# Print the versions of the main libraries to ensure consistency.
print(f"Python version: {sys.version}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
print(f"scikit-learn version: {sklearn_version}")
print(f"matplotlib version: {plt.matplotlib.__version__}")
print(f"joblib version: {joblib.__version__ if hasattr(joblib, '__version__') else 'Not Available (likely bundled)'}")
if _PLOTLY_AVAILABLE:
    print(f"plotly version: {plotly.__version__}")
else:
    print("plotly not available.")

print("\nSetup complete: Libraries imported, seed set, display configured, versions checked.")

Python version: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
pandas version: 2.2.2
numpy version: 2.0.2
scikit-learn version: 1.6.1
matplotlib version: 3.10.0
joblib version: 1.5.2
plotly version: 5.24.1

Setup complete: Libraries imported, seed set, display configured, versions checked.


In [6]:
# This cell sets up the environment for our data analysis and modeling.

# 1. Import Libraries
# We import the necessary libraries for data manipulation, visualization, and machine learning.
# pandas is for data handling and analysis.
import pandas as pd
# numpy is for numerical operations.
import numpy as np
# matplotlib.pyplot is for creating static plots.
import matplotlib.pyplot as plt
# plotly.express is for creating interactive plots easily.
import plotly.express as px
# We import specific modules from scikit-learn for machine learning tasks.
from sklearn import __version__ as sklearn_version
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, roc_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SequentialFeatureSelector
import joblib # For saving and loading models

# Check if plotly is available, otherwise set to None for version check
try:
    import plotly
    _PLOTLY_AVAILABLE = True
except ImportError:
    _PLOTLY_AVAILABLE = False
    plotly = None


# 2. Reproducibility
# Setting a fixed random seed ensures that our results are reproducible.
# This is important for competitions and for comparing different model runs.
SEED = 474
np.random.seed(SEED)
import random
random.seed(SEED)
# Some scikit-learn functions also benefit from a global random state.
# This is often handled within specific function calls, but setting it globally where possible is good practice.
# For pipelines and models, we will pass the random_state parameter explicitly.

# 3. Display Settings
# Configure pandas options to display more rows and columns.
# This helps when inspecting dataframes.
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000) # Adjust display width

# 4. Version Check
# Print the versions of the main libraries to ensure consistency.
# print(f"Python version: {sys.version}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
print(f"scikit-learn version: {sklearn_version}")
print(f"matplotlib version: {plt.matplotlib.__version__}")
print(f"joblib version: {joblib.__version__ if hasattr(joblib, '__version__') else 'Not Available (likely bundled)'}")
if _PLOTLY_AVAILABLE:
    print(f"plotly version: {plotly.__version__}")
else:
    print("plotly not available.")

print("\nSetup complete: Libraries imported, seed set, display configured, versions checked.")

pandas version: 2.2.2
numpy version: 2.0.2
scikit-learn version: 1.6.1
matplotlib version: 3.10.0
joblib version: 1.5.2
plotly version: 5.24.1

Setup complete: Libraries imported, seed set, display configured, versions checked.


## Step 1 — Data loading and sanity checks

In this section, we prepare the foundation for our predictive modeling workflow by **safely loading and validating the input datasets** (`train.csv` and `test.csv`) and by defining which variables will be used as features.

This ensures a **clean, reproducible, and competition-compliant dataset setup** before moving into exploratory analysis and model development.


### **AI-prompt:**

Create a Python workflow that prepares tabular datasets for a binary classification task in a machine learning project. The workflow must include the following capabilities:

1. **Safe data loading**: Read CSV files for training and testing datasets. If a file does not exist or is empty, raise an informative error.
2. **Target validation**: Ensure that the outcome variable (e.g., `"Exited"`) is present in the training dataset but absent in the test dataset. Confirm that it is strictly binary (0/1).
3. **Identifier detection**: Automatically detect and exclude identifier columns that should not be used as features. Use a default identifier name if available (e.g., `"CustomerId"`) or otherwise infer candidates based on high uniqueness ratios.
4. **Blocklist exclusions**: Define a set of columns (e.g., names or IDs) that must never be used in modeling or exploratory data analysis. Apply these exclusions consistently to both train and test sets.
5. **Feature/target separation**: Produce clean training features (`X`), the outcome variable (`y`), and aligned test features (`X_test`) after applying all exclusions.
6. **Data summary reports**: Generate summary tables for train and test datasets, showing column data types, counts of missing values, and unique value counts. Display the shapes of the datasets.
7. **Class balance check**: Calculate and display the distribution and proportions of the target classes to highlight potential imbalance.
8. **Final reporting**: Clearly print which columns were excluded, and report the final number of usable features in both training and test datasets.

**Additional Requirements:**

* Provide **clear, step-by-step comments** throughout, written in plain English, assuming the reader is not familiar with Python.
* Write the solution as a single, runnable notebook-ready script.
* Ensure that all error handling is informative and beginner-friendly.
* End with a concise printout confirming dataset readiness for downstream modeling.



## Step 2 — Exploratory Data Analysis (EDA)

In this section, we conduct a structured exploratory analysis of the training dataset to better understand the variables and their relationship with the target outcome (`Exited`). The goal is to identify patterns, potential data quality issues, and relationships that will inform feature engineering and model selection.

Key steps include:

* **Column classification**: Separate variables into numeric and categorical groups.
* **Target distribution**: Visualize the class balance of `Exited` to check for imbalance.
* **Univariate analysis**:

  * Numeric variables: Histograms and boxplots to examine distributions, skewness, and potential outliers.
  * Categorical variables: Bar charts to display frequency counts across categories.

* **Bivariate analysis with the target**:

  * Numeric vs. target: Boxplots comparing distributions for `Exited = 0` vs. `Exited = 1`.
  * Categorical vs. target: Stacked bar charts of exit rates by category.

* **Correlation analysis**:

  * Heatmap of correlations among numeric features.
  * Identification of highly correlated feature pairs (|r| ≥ 0.8).
  
* **Mutual information**: Quantify the strength of association between each feature and the target, highlighting the top predictors.

This analysis provides a **data-driven foundation** for subsequent preprocessing and modeling, ensuring that feature choices are informed by both statistical properties and predictive potential.



### **AI-Prompt:**

Create a Python workflow for exploratory data analysis (EDA) in a binary classification project. Assume the dataset has both numeric and categorical variables, and the target variable is called `"Exited"`.

**Requirements:**

1. **Column classification**: Automatically separate numeric and categorical variables.

2. **Target distribution**: Plot the class distribution of the target to check for imbalance.

3. **Univariate analysis**:

   * For numeric variables, create histograms and boxplots to understand distributions, skewness, and outliers.
   * For categorical variables, create bar charts showing frequency counts for each category.

4. **Bivariate analysis with target**:

   * For numeric variables, show boxplots comparing the distributions of features across the two target classes.
   * For categorical variables, generate stacked bar charts showing exit rates (proportions of the target) by category.

5. **Correlation analysis**:

   * Compute and display a correlation heatmap for numeric features.
   * Identify pairs of features with very high correlations (absolute correlation ≥ 0.8) and list them.

6. **Mutual information**: Estimate the predictive power of features by computing mutual information scores with the target. Display the top features ranked by their score.

7. **User friendliness**:

   * Assume I am not familiar with Python.
   * Provide clear and detailed comments in plain English explaining what each step does, why it is important, and how to interpret the results.
   * Ensure all plots have informative titles, labels, and readable formatting.

8. **Output**: Write the code as a single, notebook-ready workflow, returning both visualizations and summary tables where relevant.

**Goal:** The reader should clearly understand the characteristics of the dataset, the behavior of each variable, and the relationships between features and the target variable.


## Step 3 — Data Preparation and Feature Engineering



This section builds a **reproducible preprocessing pipeline** that prepares raw features for modeling and documents the transformation results. It standardizes numeric variables, encodes categoricals with a cross-version–safe One-Hot Encoder, and produces a before/after audit to verify correctness and prevent leakage into cross-validation.

**What can we potentially do here?**

* **Robust One-Hot Encoding:** Uses a compatibility wrapper (`build_ohe`) to support both `sparse_output=False` (new sklearn) and `sparse=False` (older sklearn); applies `drop='first'` to avoid the dummy-variable trap.
* **Numeric pipeline:** Median imputation → standardization (`StandardScaler`).
* **Categorical pipeline:** Most-frequent imputation → one-hot encoding (with unknown categories safely ignored).
* **Unified transformer:** `ColumnTransformer` combines numeric and categorical pipelines; non-selected columns are dropped.
* **Feature name tracing:** Derives **post-transform feature names** (including expanded one-hot columns) for transparent modeling and diagnostics.
* **Before/After audit (no leakage):** Clones and fits the preprocessor **only for reporting**, then:

  * Reports input shape, dtype mix, and missing-value profile.
  * Shows transformed matrix shape and total engineered feature count.
  * Audits each categorical: total levels, encoded columns kept, and the **dropped baseline**.
  * Optionally previews the transformed design matrix (first rows) as a DataFrame.

**Why it matters?**

* Ensures **consistent, CV-safe** preprocessing across all models.
* Prevents **data leakage** by separating reporting from the training pipeline.
* Delivers **traceability** of engineered features (names, counts, baselines).
* Creates a **clean, model-ready design matrix** that downstream estimators can consume without additional wrangling.

> Note: At this stage, “feature engineering” focuses on **systematic transformations** (imputation, scaling, encoding). Domain-specific feature creation (e.g., interactions, ratios, buckets) can be added later as an extension to this pipeline.

> **Data leakage** occurs when information from outside the training process—especially from the validation or test sets—unintentionally influences model training. This leads to overly optimistic performance estimates that do not generalize to new data. In preprocessing pipelines, leakage often happens when imputers, scalers, or encoders are fit on the full dataset before cross-validation, allowing statistical properties of the validation folds (e.g., mean, median, category levels) to “leak” into the model. To prevent this, transformations must be fit **only on the training portion of each fold** and then applied to the corresponding validation data. In this section, the code avoids leakage by cloning and fitting the preprocessor strictly for reporting purposes, while the actual training pipeline applies the same steps safely within cross-validation.



### **AI-Prompt:**

Design a Python workflow for **data preparation and feature engineering** in a binary classification project. The workflow must be structured as a reusable pipeline and include the following capabilities:

1. **Numeric feature handling**

   * Impute missing values using the median.
   * Standardize features so they are on a comparable scale.

2. **Categorical feature handling**

   * Impute missing values with the most frequent category.
   * Apply one-hot encoding, dropping one baseline level per variable to avoid the dummy-variable trap.
   * Ensure compatibility across different versions of scikit-learn (i.e., handle environments where encoder parameters may differ).

3. **Unified preprocessing pipeline**

   * Use a column transformer to combine numeric and categorical pipelines.
   * Drop any columns not explicitly included in the transformations.

4. **Feature name tracking**

   * Extract and report the transformed feature names (important for one-hot encoded variables that expand into multiple columns).

5. **Before/After overview function**

   * Print dataset shape, number of numeric and categorical features, data type counts, and missing value summary before preprocessing.
   * Apply a cloned version of the pipeline (for reporting only) to show the transformed dataset shape and feature count after preprocessing.
   * Provide an audit of categorical encoding, including total levels, number of encoded columns, and which baseline was dropped.
   * Display a preview of the transformed design matrix (first few rows) with feature names.
   * Ensure this audit does not cause data leakage into model training or cross-validation.

6. **User friendliness**

   * Assume I am not familiar with Python.
   * Provide **clear, plain-English comments** that explain each step, why it is needed, and how to interpret the output.
   * Include informative printouts for transparency.

**Goal:** Produce a single, notebook-ready script that prepares numeric and categorical features for machine learning in a robust, transparent, and reproducible way, while preventing data leakage and ensuring interpretability of engineered features.


## Step 4 — Basic Model

In this step, we fit a **baseline predictive model** using **logistic regression** and evaluate its performance with rigorous cross-validation and bootstrap confidence intervals. The purpose is to establish a benchmark model that is simple, interpretable, and statistically well-documented before moving to more complex methods.

**Key components of this step**:

* **Cross-validation with stratification**: A 5-fold stratified cross-validation ensures that class proportions are preserved in each fold, providing a reliable estimate of model performance.

* **Out-of-Fold (OOF) predictions**: Predictions for all observations are generated from models trained without those observations, allowing us to estimate performance as if on unseen data.

* **Evaluation metrics**: For each fold and overall, we compute:

  * **Accuracy** and **misclassification error** (1 – accuracy).

  * **ROC AUC** (area under the receiver operating characteristic curve) as a robust measure of ranking ability.

* **Bootstrap confidence intervals**: Using stratified bootstrap resampling, we compute 95% confidence intervals for both AUC and error rates, giving a sense of uncertainty around the performance metrics.

* **Fold-level summary**: Reports mean and standard deviation of AUC and error across folds, as well as a detailed per-fold metrics table.

* **Pipeline integration**: The preprocessing steps defined earlier (imputation, scaling, one-hot encoding) are seamlessly integrated into the logistic regression model through a scikit-learn pipeline, ensuring that all data transformations are applied consistently during training and evaluation.

This structured evaluation provides a **transparent and statistically sound baseline** against which more advanced models can later be compared.



### **AI-Prompt:**

Create a Python workflow for building and evaluating a **baseline predictive model** using logistic regression. The workflow should be implemented with a reproducible machine learning pipeline and include the following capabilities:

1. **Pipeline integration**

   * Incorporate the preprocessing steps already defined (numeric imputation + scaling, categorical imputation + one-hot encoding).
   * Ensure transformations are applied consistently during both training and validation.

2. **Model training**

   * Use logistic regression as the classifier.
   * Handle potential class imbalance by applying appropriate weighting.

3. **Cross-validation**

   * Apply stratified k-fold cross-validation (e.g., 5 folds) to maintain class proportions in each split.
   * Generate **out-of-fold (OOF) predictions** for every observation to provide unbiased performance estimates.

4. **Evaluation metrics**

   * For each fold and overall, compute **accuracy**, **misclassification error**, and **ROC AUC**.
   * Report both per-fold metrics and aggregated summaries (mean and standard deviation across folds).

5. **Uncertainty estimation**

   * Use **bootstrap resampling** (stratified by class) to calculate 95% confidence intervals for AUC and misclassification error.

6. **Outputs and reporting**

   * Print overall OOF metrics (AUC, accuracy, error).
   * Display bootstrap confidence intervals.
   * Provide a summary of fold-level metrics and a formatted table of per-fold results.

**Additional requirements**

* Assume I am not familiar with Python—include **step-by-step comments in plain English** explaining what each part of the workflow does, why it is necessary, and how to interpret the results.
* Use clear, beginner-friendly printouts to highlight the main results.

**Goal:** Deliver a notebook-ready, transparent, and statistically sound baseline evaluation of logistic regression that can serve as a benchmark for comparing more advanced models later.


## Step 5 — More Complex Model

Building on the baseline logistic regression, this step explores **more flexible predictive models** that can capture non-linear relationships, interactions between variables, and complex decision boundaries. The goal is to assess whether more advanced algorithms can improve predictive performance beyond the benchmark established in Step 4.

**Key components of this step**:

* **Model selection**: Introduce and train models with greater representational power, such as decision trees, random forests, gradient boosting (e.g., XGBoost, LightGBM), or regularized regression methods (LASSO, Ridge, Elastic Net).
* **Cross-validation evaluation**: Apply the same rigorous 5-fold stratified cross-validation procedure to ensure comparability with the baseline model.
* **Out-of-Fold (OOF) predictions**: Generate unbiased performance estimates by aggregating predictions from models trained without the corresponding observations.
* **Performance metrics**: Compute and compare accuracy, misclassification error, and ROC AUC for each model. Use per-fold results, overall metrics, and summary statistics (mean and standard deviation across folds).
* **Uncertainty estimation**: Extend evaluation with bootstrap confidence intervals for AUC and error, quantifying the reliability of improvements relative to the baseline.
* **Pipeline integration**: Ensure preprocessing (imputation, scaling, encoding) is consistently applied within each model’s pipeline, preventing leakage and preserving comparability.
* **Model comparison**: Present results side by side with the baseline, highlighting both performance gains and potential trade-offs (e.g., interpretability vs. accuracy).

This step provides a **systematic framework for testing more advanced algorithms** while maintaining the same statistical rigor, making it possible to judge whether added model complexity delivers meaningful improvements.



## Step 5.1: Forward stepwise feature selection

### **AI-Prompt:**

Design a Python workflow for performing **forward stepwise feature selection** in a binary classification project. The workflow should use logistic regression as the base model and follow these requirements:

1. **Baseline model**

   * Build a logistic regression classifier with reasonable defaults (e.g., balanced class weights, sufficient iterations, reproducible random seed).
   * Integrate preprocessing (numeric imputation + scaling, categorical imputation + one-hot encoding) into a scikit-learn pipeline.
   * Fit the baseline pipeline to confirm feature compatibility and extract the full list of transformed feature names after preprocessing.

2. **Sequential Feature Selection (SFS)**

   * Apply forward stepwise selection using logistic regression as the estimator.
   * Use cross-validation (e.g., 5 folds) and **ROC AUC** as the evaluation metric to decide which features to keep.
   * Automatically determine the optimal number of features.
   * Record the indices and names of the selected features.

3. **Refined pipeline**

   * Build a new pipeline that applies preprocessing, then restricts the dataset to the selected features, and finally fits logistic regression.
   * Ensure the pipeline can be reused seamlessly for training, validation, and later prediction.

4. **Model evaluation**

   * Re-evaluate the refined pipeline using cross-validation.
   * Generate **out-of-fold (OOF) predictions** and compute:

     * Accuracy
     * Misclassification error
     * ROC AUC
   * Summarize fold-level results (mean and standard deviation of AUC across folds).
   * Compute **95% bootstrap confidence intervals** for AUC and error to quantify uncertainty.

5. **Reporting**

   * Print the number of features selected and list the top selected feature names.
   * Report OOF AUC and error, along with their bootstrap confidence intervals.
   * Report mean and standard deviation of fold-level AUC.
   * Display a formatted per-fold metrics table for accuracy, error, and AUC.

**Additional requirements:**

* Assume I am not familiar with Python. Include **clear, plain-English comments** at every step to explain what is happening, why it is important, and how to interpret the results.
* Output should be a single, notebook-ready workflow that is reproducible and easy to extend.

**Goal:** Produce a transparent and statistically rigorous stepwise feature selection process that demonstrates how model performance changes when using only the most predictive subset of features.



## Step 5.2: Lasso Regression



### **AI-Prompt:**

Create a Python workflow to tune and evaluate a **LASSO (L1-penalized) logistic regression** for a binary classification task. The workflow should be **pipeline-based**, reproducible, and beginner-friendly, with clear comments explaining every step and why it matters.

**Requirements**

1. **Setup & Reproducibility**

   * Assume a preexisting preprocessing pipeline (numeric imputation+scaling; categorical imputation+one-hot) and training data `X, y`.
   * Use a fixed random seed throughout.
   * Use class weighting to address potential imbalance.

2. **Regularization grid**

   * Define a logarithmic grid of **λ (lambda)** values spanning several orders of magnitude.
   * Recall that scikit-learn’s logistic regression uses **C = 1/λ** (larger C ⇒ weaker regularization).
   * For each λ, configure an **L1-penalized** logistic regression with an appropriate solver and sufficient iterations.

3. **Cross-validated evaluation per λ**

   * Wrap model + preprocessing in a single pipeline.
   * Run **stratified k-fold cross-validation** (e.g., 5 folds).
   * Collect **out-of-fold (OOF)** predictions and compute:

     * **ROC AUC** (primary metric)
     * **Misclassification error** (1 − accuracy)
   * Also compute **fold-wise mean and standard deviation** of AUC.

4. **Uncertainty quantification**

   * For each λ, compute **95% bootstrap confidence intervals** (stratified by class) for **AUC** and **error**, using OOF predictions.

5. **Model sparsity diagnostic**

   * (Reporting only) Fit the L1 model on the **full training set** for each λ and record the **number of non-zero coefficients** to quantify sparsity.
   * Clarify in comments that this full-fit is **not** used for CV metrics (to avoid leakage) and is for interpretability reporting only.

6. **Results aggregation & selection**

   * Build a results table (DataFrame) with, for each λ: λ, C, OOF AUC, OOF error, their 95% CIs, mean CV AUC, std CV AUC, and non-zero coefficient count.
   * Select the **best λ** by **highest OOF AUC** (if tied, prefer the more regularized model).
   * Clearly print the chosen λ and C and summarize the corresponding metrics and CIs.

7. **Visualization**

   * Produce a **semilog plot** of performance vs. λ showing at least:

     * OOF AUC across λ
     * Mean CV AUC across λ
     * A vertical line marking the **best λ**
   * Use informative titles, axis labels, and a legend.

8. **Beginner-friendly guidance**

   * Add **plain-English comments** explaining:

     * What L1 regularization does (sparsity/feature selection).
     * Why we evaluate with OOF predictions and CV.
     * Why we use bootstrap CIs (quantify uncertainty).
     * How to read the results table and the λ vs. AUC plot.
   * End with a short textual summary interpreting the selected λ, expected generalization performance, and sparsity trade-offs.

**Output:**
A single, notebook-ready workflow that runs end-to-end, prints the results table and key metrics, shows the plot, and provides clear commentary suitable for readers new to Python and model tuning.


## Step 6 — Compare models with visualization

### **AI-Prompt:**

Create a Python visualization and reporting workflow that **compares multiple binary classification models** using standardized evaluation outputs. Assume you are given four result objects—one each for: **Full (Logistic Regression)**, **Forward Stepwise**, **LASSO (best λ)**, and **GAM**—all with the same structure. The workflow must be beginner-friendly and include clear, plain-English comments explaining every step, what it does, why it matters, and how to read the output.

**Inputs (assume already available in memory):**

* `results_full`, `results_stepwise`, `results_lasso_best`, `results_gam` — each is a dict containing:

  * `overall`: `{"roc_auc": float, "misclassification_error": float}`
  * `bootstrap_ci`: `{"auc_ci": (low, high), "error_ci": (low, high)}`
  * Either `fold_summary` with `{"roc_auc_mean": float, "roc_auc_std": float}` **or** `fold_metrics` (list of per-fold dicts with `"roc_auc"`).
* Optional diagnostics:

  * `lasso_best_row` (dict) with keys like `"lambda"`, `"C"`, `"nonzero_coef"` for sparsity and hyperparameter reporting.
  * `selected_feature_names` (list) for Stepwise to report how many transformed features were selected.

**Requirements:**

1. **Metric assembly**

   * Build a consistent comparison across models: **OOF ROC AUC**, **OOF Misclassification Error**, and their **95% bootstrap CIs**.
   * Derive **Mean CV AUC** and **Std CV AUC** either from `fold_summary` or by computing them from `fold_metrics`.

2. **Plot 1 — AUC with uncertainty**

   * Create a bar chart of **OOF ROC AUC** for the four models.
   * Add **95% CI error bars** on each bar.
   * Overlay **mean CV AUC ± 1 SD** as points with vertical error bars to convey fold-to-fold stability.
   * Annotate each bar with its AUC value (3 decimals).
   * Include a clear title, axis labels, and legend.

3. **Plot 2 — Misclassification Error with uncertainty**

   * Create a bar chart for **OOF Misclassification Error** with **95% CI error bars**.
   * Annotate each bar with its value (3 decimals) and include a descriptive title and y-axis label.

4. **Tabular summary**

   * Build a formatted table (DataFrame) with one row per model and columns:

     * `OOF AUC`, `AUC 95% CI (low)`, `AUC 95% CI (high)`
     * `OOF Error`, `Error 95% CI (low)`, `Error 95% CI (high)`
     * `Mean CV AUC`, `Std CV AUC`
   * If available, add optional columns:

     * For LASSO: `Best λ (1/C)`, `Best C`, `Non-zero Coefs` (sparsity).
     * For Stepwise: `Selected Features` (length of `selected_feature_names`).
   * Format numeric columns to 4 decimals (6 for λ and C). Display the styled table.

5. **Beginner-friendly guidance**

   * Comment on:

     * The difference between **OOF** vs. **mean CV** metrics and why both are shown.
     * What 95% CIs indicate and how to compare overlapping intervals.
     * How **Std CV AUC** reflects stability across folds.
     * Why sparsity (non-zero coefficients) and feature counts aid interpretability.

6. **Clean API**

   * Provide a single callable function (e.g., `plot_model_comparison_extended(...)`) that accepts the four result dicts and the two optional diagnostics.
   * The function should **display both plots and the table**, with no side effects beyond visualization/printing.

**Output:**
A single, notebook-ready workflow that produces the two comparison plots and the formatted summary table, with clear commentary suitable for readers new to Python and model evaluation.


## Step 7 — Model selection and rationale

This section implements a **transparent, criteria-driven selector** that chooses the best model among four candidates—**Full Logistic Regression**, **Forward Stepwise**, **LASSO (best λ)**, and **GAM**—and returns both the **winner** and a **human-readable Markdown rationale** summarizing the decision.

**What it does**

* **Standardizes inputs:** Reads each model’s evaluation bundle (OOF AUC, OOF error, 95% bootstrap CIs; plus either fold-level summaries or raw fold metrics). If fold summaries are missing, it **recomputes mean/SD AUC from per-fold scores**.
* **Primary criterion (performance):** Ranks models by **Out-of-Fold ROC AUC**. If the top two are close (within a small tolerance) **and** their **95% CIs overlap**, the code avoids over-interpreting negligible differences.
* **Stability tie-breakers:** When AUCs are statistically similar, it prefers **higher mean CV AUC**, then **lower CV AUC standard deviation** (more stable across folds), then **lower OOF misclassification error**.
* **Parsimony final tie-break:** If metrics remain effectively tied, it favors simpler models in this order: **LASSO** (fewest non-zero coefficients), **Forward Stepwise** (fewest selected features), **Full**, **GAM**.
* **Clear rationale:** Generates a concise Markdown report listing, for every model, the **OOF AUC with 95% CI**, **Mean±SD CV AUC**, and **OOF error**, plus a brief **decision note** explaining why the chosen model won. When available, it appends **sparsity** (LASSO non-zero coefficients and λ/C) and **feature count** (Stepwise).

**Why it matters**

* Balances **point performance, uncertainty, and stability**, not just a single score.
* Reduces **false wins** from tiny numeric fluctuations by using **tolerances** (`eps_*`) and **CI overlap** checks.
* Encourages **interpretability** through parsimony when models are neck-and-neck.
* Produces a **reproducible, report-ready justification** you can paste into notebooks or competition write-ups.


### **AI-Prompt:**

Design a **model selection workflow** that chooses the best classifier among four candidates—**Full Logistic Regression**, **Forward Stepwise**, **LASSO (best λ)**, and **GAM**—using standardized evaluation results already computed elsewhere. Write it so a beginner can follow: include clear, plain-English comments explaining every step, why it’s done, and how to interpret the output.

**Inputs (assumed available in memory):**

* Four result objects (dict-like), one per model: `results_full`, `results_stepwise`, `results_lasso_best`, `results_gam`.

  * Each contains:

    * `overall`: `{"roc_auc": float, "misclassification_error": float}`
    * `bootstrap_ci`: `{"auc_ci": (low, high), "error_ci": (low, high)}`
    * Either `fold_summary` with `{"roc_auc_mean": float, "roc_auc_std": float}` **or** `fold_metrics` (list with per-fold `"roc_auc"`).
* Optional diagnostics:

  * `lasso_best_row` (dict) with `{"lambda": float, "C": float, "nonzero_coef": int}` for sparsity/hyperparam reporting.
  * `selected_feature_names` (list) from forward stepwise to report how many transformed features were selected.

**What to build:**

1. A single function (e.g., `select_model(...) -> (choice_key: str, rationale_markdown: str)`) that implements a **clear decision policy**:

   **Primary criterion:**

   * Choose the model with the **highest Out-of-Fold (OOF) ROC AUC**.
   * Treat tiny numerical differences as ties using a configurable tolerance `eps_auc`.
   * If the top two AUCs are close **and** their **95% AUC CIs overlap**, do not assume a real difference.

   **If AUCs are close and CIs overlap, use tie-breakers in order:**

   * Higher **mean CV AUC** (prefer better average performance across folds).
   * Lower **CV AUC standard deviation** (prefer more stable performance).
   * Lower **OOF misclassification error**.

   **Final parsimony tiebreak (only if metrics are effectively tied):**

   * Prefer models that are simpler/sparser in this order: **LASSO** (fewest non-zero coefficients), **Forward Stepwise** (fewest selected features), **Full Logistic**, **GAM**.
   * Only invoke this if all metrics are within tiny configurable thresholds of the current top model.

2. Helper behavior:

   * If `fold_summary` is missing, compute mean and std of per-fold AUC from `fold_metrics`.
   * Implement a small utility to check **confidence-interval overlap**.
   * Make tolerances configurable: `eps_auc`, `eps_cv`, `eps_sd`, `eps_err`.

3. Output & explanation:

   * Return the chosen model key (e.g., `"lasso"`, `"forward_stepwise"`, `"full"`, `"gam"`).
   * Return a **Markdown rationale** that concisely reports, for each model:

     * **OOF AUC** with its **95% CI**
     * **Mean±SD CV AUC**
     * **OOF error**
     * Decision note explaining **why** the winner was selected (e.g., “highest OOF AUC with non-overlapping CI” or “AUCs similar; selected higher/stabler CV AUC and lower OOF error”).
   * If available, append **parsimonious details** (e.g., “LASSO non-zero coefficients: N; best λ (1/C)=…, C=…”, “Stepwise selected features: K”).

**User-friendly requirements:**

* Assume I’m new to Python.
* Add **step-by-step comments** explaining: OOF vs CV metrics, why CIs matter, how tie-breakers work, why parsimony helps interpretability.
* Keep function I/O clean and safe (handle missing pieces gracefully).

**Deliverables:**

* A **notebook-ready** function implementing the policy above.
* A short usage example that calls the function with the four result objects (and optional diagnostics), prints the chosen model key, and prints the rationale.

**Goal:**
Provide a transparent, statistically sensible, and reproducible **model selection routine** that balances point performance, uncertainty, stability, and simplicity—ready to drop into a competition or classroom workflow.

## Step 8 — Final training on the full training data

In this section we **finalize the model** selected in the previous step (one of **Full Logistic Regression**, **Forward Stepwise**, **LASSO (best λ)**, or **GAM**) and train it on the **entire training set** to maximize learning before producing competition/test predictions. The code first builds the correct pipeline for the chosen model, then fits it end-to-end with the same preprocessing used during evaluation to avoid any mismatch or leakage.

To aid **traceability and reproducibility**, we compute how many **transformed features** the pipeline actually uses. A small helper safely clones and fits the preprocessor on `X` purely for reporting, returning the transformed feature count (and names when available). For stepwise models, we report the number of **selected** transformed features; for LASSO/GAM/Full, we report the full transformed dimensionality.

After training, the code:

* **Persists** the fitted pipeline to `final_model.joblib` (ready for downstream scoring).
* Writes a compact **metadata JSON** (`final_model_metadata.json`) capturing the timestamp, seed, selected model, feature count, an optional preview of feature names, and model-specific diagnostics (e.g., LASSO’s best λ/C and non-zero coefficient count, Stepwise’s number of selected features).
* Prints friendly confirmations, including the **selection rationale** if available, so the final choice is easy to justify in reports.

This makes the notebook **deployment-ready**: you end with a single serialized model plus auditable metadata that documents exactly **what** was trained, **how**, and **why**.


### **AI-Prompt:**

Create a **final training and persistence workflow** for a binary classification project that selects among four already-evaluated candidates—**Full Logistic Regression**, **Forward Stepwise**, **LASSO (best λ)**, and **GAM**—and then fits the chosen pipeline on the **entire training set**. The workflow must be notebook-ready and heavily commented in plain English so a newcomer can follow every step.

**Assume these objects already exist in memory:**

* `model_choice` (one of: `"full"`, `"forward_stepwise"`, `"lasso"`, `"gam"`)
* Preprocessing and pipelines: `preprocessor`, `full_pipeline`, `refined_pipeline` (stepwise), `gam_pipeline`
* Data: `X` (features as DataFrame), `y` (binary target as Series), `SEED` (int)
* Stepwise artifacts: `selected_idx` (list of selected column indices), `selected_feature_names` (list; may be absent)
* LASSO artifacts: `best_row` (row/dict with at least `C`, optionally `lambda`, `nonzero_coef`)
* Optional text: `rationale_text` (Markdown string explaining why the model was chosen)
* Utility: `get_transformed_feature_names(preprocessor_like)` that returns names after preprocessing

**What the workflow must do (with clear comments):**

1. **Helper to count transformed features**

   * Build a small function that **clones and fits** the provided `preprocessor` on `X`, then returns:

     * The **number of transformed features**.
     * The **feature names** if available; otherwise `None`.
   * Explain why cloning/fitting just for counting is safe and does **not** affect training.

2. **Construct the final pipeline based on `model_choice`**

   * `"full"`: use the already-defined `full_pipeline`. Try to get transformed feature names from an existing fitted preprocessor if available; otherwise call the helper.
   * `"forward_stepwise"`: use `refined_pipeline` (preprocess → selector → classifier). Feature count equals `len(selected_idx)`. Prefer `selected_feature_names` if available.
   * `"lasso"`: rebuild a final pipeline using **L1 logistic regression** with **C** from `best_row["C"]` (remind that `C = 1/λ`). Use the shared `preprocessor`. Count features via the helper.
   * `"gam"`: use the already-defined `gam_pipeline`; count features via the helper.
   * Validate `model_choice` and raise a clear error if it’s unknown.

3. **Fit on the entire training data**

   * Fit the chosen pipeline on `X, y`.
   * Explain why we now train on **all** data (after model selection), and caution about reusing any cross-validation objects (we are not).

4. **Persist the trained model**

   * Save the fitted pipeline to `final_model.joblib` using a standard serialization library.
   * Print a friendly confirmation message.

5. **Persist lightweight metadata (JSON)**

   * Write `final_model_metadata.json` capturing:

     * `timestamp` in UTC ISO 8601 with “Z” suffix
     * `seed`
     * `model_choice`
     * `feature_count`
     * `selected_features_preview` (first ~25 names if available; otherwise `null`)
     * `rationale` (if provided)
     * If winner is LASSO: include `lambda` (if available), `C`, and `nonzero_coef` (if available)
     * If winner is Stepwise: include `n_selected`
   * Explain why metadata aids **reproducibility** and quick auditing.

6. **User-facing printouts**

   * Print the save location for the model, the number of transformed features used, and (if present) the selection rationale text.

7. **Beginner notes & safety**

   * Add comments clarifying common pitfalls (e.g., not leaking CV artifacts, why we re-fit on full training data).
   * Be defensive with missing optional fields (`selected_feature_names`, `best_row["lambda"]`, `best_row["nonzero_coef"]`) and handle gracefully.

**Output:**

* A single, well-commented, notebook-ready block that:

  * Builds the correct final pipeline based on `model_choice`
  * Fits on `X, y`
  * Saves `final_model.joblib`
  * Saves `final_model_metadata.json` with the fields above
  * Prints concise confirmations and the feature count

**Tone & style:**

* Plain-English comments at each step.
* Short explanations of *why* each action is taken (e.g., cloning preprocessor, re-fitting on all data, what metadata fields mean).
* Keep variable names readable and consistent with the assumptions above.



## Step 9 — Predict on the test set and export CSV

This section loads the **final trained pipeline**, aligns the **test set schema** to what the pipeline’s preprocessor expects, generates **class-1 probabilities**, and writes a **competition-ready `submission.csv`** with a robust ID column.


**Key Steps**

1. **Model Loading**
   The persisted predictive model pipeline is reloaded from disk. This guarantees that the exact trained model is used consistently at inference time. A fail-fast check ensures the file exists before proceeding.

2. **Test Data Preparation**
   The test dataset is loaded, and the script automatically detects the identifier column. Because identifier names may vary (`id`, `CustomerId`, `CostumerId`, etc.), the code applies a detection routine that searches for expected variants. If no known name is found, it falls back on a heuristic that selects a column with unique, non-missing values.

3. **Identifier Integrity**
   The detected identifier column is re-read from the raw file with an enforced string type. This prevents accidental corruption of identifiers, such as conversion to floats, loss of leading zeros, or display in scientific notation. The identifier is then normalized under the standard name `id`.

4. **Feature Matrix Construction**
   All non-feature columns (`id`, potential variants, and any target columns like `Exited`) are dropped, leaving only predictors for the model. The pipeline’s preprocessing step is then used to align the test features with the expected schema. Missing columns are filled with `NaN`, and the column order is matched precisely.

5. **Prediction**
   The pipeline generates predicted probabilities for the positive outcome class. If the model supports probability prediction (`predict_proba`), the probabilities are taken directly. Otherwise, the script falls back to a decision function and applies a logistic transformation.

6. **Submission File Assembly**
   A submission DataFrame is created with exactly two columns:

   * `id` (the preserved identifiers from the test file)
   * `Exited` (the predicted probability of exit)

   The file is written to disk without altering row order, ensuring alignment between identifiers and predictions. Sanity checks validate that the number of rows in the submission matches the test set.

7. **Diagnostics and Validation**
   Additional routines confirm the correctness of the submission:

   * Schema verification (only `id` and `Exited`).
   * Count comparison between test and submission.
   * Set-level checks to confirm that every identifier in the test appears in the submission and vice versa.
   * Positional alignment checks to confirm that identifiers match row-for-row.
   * Reporting of discrepancies, including mismatched IDs or indices, to help pinpoint and resolve issues.

**Purpose and Rationale**

This robust submission workflow ensures that the exported predictions satisfy all competition requirements: correct schema, faithful preservation of test identifiers, row-by-row alignment, and valid probability ranges. By embedding validation and diagnostics, it provides a safeguard against silent errors that could otherwise invalidate results or reduce the model’s standing in leaderboard evaluations.

This section therefore functions as the critical final step of the predictive modeling pipeline, converting trained models into valid, reliable, and competition-ready outputs.


### **AI-Prompt**

Design a robust, production-grade inference and submission workflow for a supervised binary classification project that predicts the probability of the positive class for each record in a test dataset and exports a competition-ready submission file.

**Context and constraints**

* Test data are provided in a CSV file. The identifier column name may vary across datasets (e.g., id, ID, Id, CustomerId, CustomerID, customerId, CostumerId).
* Identifiers must be preserved exactly as they appear in the CSV (no numeric coercion, no scientific notation, no “.0” suffixes).
* The final submission must contain exactly two columns in this order: `id`, `Exited`, where:

  * `id` is the exact identifier from the test file (string).
  * `Exited` is the predicted probability of the positive class in [0,1], **rendered in the CSV with exactly five digits after the decimal point** (e.g., `0.12345`, `1.00000`). Trailing zeros must be preserved.
* Row order in the submission must match the row order of the input test CSV unless explicitly stated otherwise.

**Inputs**

* Path to the persisted model pipeline file.
* Path to the test CSV file.
* Desired output path for the submission CSV.

**Outputs**

1. A submission CSV with columns:

   * `id`: the exact identifier values from the test CSV (string type).
   * `Exited`: the predicted probability for the positive class (float in [0,1]) **and serialized to disk with exactly five decimal places**.
2. Console logs (or equivalent) confirming schema, counts, integrity checks, and **formatting enforcement**.
3. Optional diagnostic report to help debug identifier mismatches and **formatting violations**.

**Functional requirements**

1. Load the persisted pipeline from disk. Fail fast with a clear error if the file is missing.
2. Read the test CSV for features.
3. Detect the identifier column using the following logic:

   * Prefer a pre-defined list of common identifier names (case variants included).
   * If none are found, select a single column whose values are all non-null and unique across all rows.
   * If still ambiguous, raise a descriptive error instructing the user to supply/rename an identifier column.
4. Re-read the identifier column only, enforcing string typing at read time to guarantee lossless fidelity (no numeric casting). Treat this column as the single source of truth for identifiers throughout the workflow.
5. Build the feature matrix by dropping the identifier column and any target artifacts if present (e.g., `Exited`).
6. If the pipeline includes a preprocessing step that expects specific columns, align the test feature columns to those expectations in a column-wise manner only (preserve row order). For missing expected columns, create them with NA values; for unexpected extras, drop them.
7. Generate probability predictions for the positive class:

   * Prefer `predict_proba` (use the second column).
   * If unavailable, use `decision_function` and map scores to (0,1) via a logistic transform.
   * If neither is available, fail with a clear, actionable error.
   * Clip to [0,1] for numerical safety.
8. Construct the submission DataFrame with exactly two columns, in order: `id`, `Exited`. Keep `id` as string.
9. Do not sort unless required by the competition; preserve original file order.
10. **Persist the submission with `Exited` rendered to exactly five decimal places:**

    * Use a serialization method that preserves trailing zeros, e.g., `DataFrame.to_csv(..., float_format="%.5f")`.
    * Alternatively (language-agnostic): convert the `Exited` column to strings via fixed-point formatting, e.g., `f"{p:.5f}"`, ensuring the decimal separator is `.`.
    * Ensure no thousands separators or locale-specific formatting are introduced.
11. Write the submission to disk with index disabled.

**Validation and integrity checks (must run automatically)**

* Assert the submission has exactly two columns: `["id", "Exited"]`.
* Assert `0 ≤ Exited ≤ 1` for all rows (operate on numeric probabilities before formatting).
* Assert row counts between test and submission match.
* Compare the set of unique `id`s in the submission against those in the test CSV (read as strings). They must match exactly.
* Optionally verify positional alignment (element-wise equality of `id` in original test order vs. submission order) and print True/False.
* **Format compliance check (required):** after writing, re-read the CSV with `Exited` as string and verify every value matches the regex `^\d+\.\d{5}$`. Fail with an actionable error if any row violates the five-decimal constraint and print a few offending examples.

**Diagnostics (print concise, actionable findings)**

* If `id` sets differ, print the first few identifiers that exist only in test and only in submission.
* If positional alignment fails, print the first few mismatched row indices along with the pair of `id` values (test vs. submission).
* Echo the detected identifier column name used as the source of truth.
* **If format compliance fails,** print sample rows with the non-compliant `Exited` strings and instructions to enforce fixed-point formatting (e.g., `float_format="%.5f"` or fixed-point string formatting).

**Edge-case handling**

* Identifier column appears with unexpected capitalization or common misspelling (e.g., `CostumerId`).
* Identifier stored as large integers, floats, or strings with leading zeros; ensure the exported `id` values match the CSV exactly by enforcing string dtype at read time.
* Test CSV accidentally includes a target column; ensure it is dropped from features.
* Pipeline’s preprocessor expects columns not present in test; create missing columns as NA and align column order without altering row order.
* Estimator lacks `predict_proba`; fall back to `decision_function` with logistic mapping; otherwise raise a clear error.
* **Locale differences:** force `.` as the decimal separator and avoid thousands separators to keep the `^\d+\.\d{5}$` contract.

**Deliverables**

* Fully commented, readable implementation that performs the above workflow end-to-end.
* Inline comments that explain every step in plain language for non-experts, including why each step is necessary (e.g., “read IDs as strings to prevent scientific notation or .0 artifacts”; “use fixed-point formatting to guarantee exactly five decimals”).
* Console prints that summarize success (file path, shape, schema) and any detected anomalies.
* A short “How to run” note at the top specifying where to set the three paths (model file, test file, output file).

**Acceptance criteria**

* The submission CSV passes all validation checks, including exact set equality of `id`s and correct schema/order.
* Predictions are probabilities in [0,1], aligned row-by-row with the original test file.
* **Every `Exited` value is serialized in the CSV with exactly five digits after the decimal point (e.g., `0.12345`, `1.00000`), verified by a post-write regex check.**
* The workflow fails loudly and informatively when assumptions are violated (missing model, missing/ambiguous `id`, no probability-like outputs, **formatting non-compliance**), providing clear remediation guidance.

**Instruction to the model**

Produce the complete solution described above with thorough, plain-English comments for each step so that a reader unfamiliar with programming can understand the purpose and rationale of the operations. Include the validation and diagnostics routines, and **explicitly implement both the fixed-point five-decimal serialization of `Exited` and the post-write regex validation** to guarantee the final output matches the contract exactly.


## Step 99 — Reporting and reproducibility


**Executive summary (template):** filtered out ID-like fields and `Surname`; model comparison with bootstrap CIs; final choice via AUC → error → parsimony.

**Assumptions:** median/mode imputation; OHE handle_unknown='ignore'; LogisticRegression(class_weight='balanced'); 5-fold stratified CV; 2,000 stratified bootstrap resamples.

**How to run:**
- Place `train.csv` and `test.csv` next to the notebook.
- Run all cells.
- Artifacts: `final_model.joblib`, `submission.csv`.


In [7]:

# Write requirements.txt
req = {
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'scikit-learn': sklearn_version,
    'matplotlib': plt.matplotlib.__version__,
    'joblib': joblib.__version__ if hasattr(joblib, '__version__') else '>=1.3.0',
    'plotly': '>=5.0.0' if _PLOTLY_AVAILABLE else None
}
lines = [f"{k}=={v}" for k,v in req.items() if v is not None]
with open('requirements.txt','w') as f:
    f.write("\n".join(lines))
print('requirements.txt written:\n' + "\n".join(lines))


requirements.txt written:
pandas==2.2.2
numpy==2.0.2
scikit-learn==1.6.1
matplotlib==3.10.0
joblib==1.5.2
plotly==>=5.0.0
